In [1]:
!git clone https://github.com/Wisteria717/Chords_Dataset.git

Cloning into 'Chords_Dataset'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 60 (delta 8), reused 53 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 9.22 KiB | 1.54 MiB/s, done.
Resolving deltas: 100% (8/8), done.


In [2]:
!rm -rf /content/Chords_Dataset
!apt-get install -y git-lfs
!git lfs install
!git clone https://github.com/Wisteria717/Chords_Dataset.git
!ls -lh /content/Chords_Dataset/WavFiles

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  git-lfs
0 upgraded, 1 newly installed, 0 to remove and 33 not upgraded.
Need to get 3,908 kB of archives.
After this operation, 11.7 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 git-lfs amd64 3.4.1-1ubuntu0.4 [3,908 kB]
Fetched 3,908 kB in 3s (1,488 kB/s)
Selecting previously unselected package git-lfs.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../git-lfs_3.4.1-1ubuntu0.4_amd64.deb ...
Unpacking git-lfs (3.4.1-1ubuntu0.4) ...
Setting up git-lfs (3.4.1-1ubuntu0.4) ...
Processing triggers for man-db (2.12.0-4build2) ...
Git LFS initialized.
Cloning into 'Chords_Dataset'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 60 (delta 8)

In [3]:
%cd /content/Chords_Dataset
!git lfs pull
%cd /content
!ls -lh /content/Chords_Dataset/WavFiles

/content/Chords_Dataset
/content
total 227M
-rw-r--r-- 1 root root 8.2M Nov 13 11:30 'FirstBatch - MajorC2-B3_Organ.wav'
-rw-r--r-- 1 root root 8.1M Nov 13 11:30 'FirstBatch - MajorC2-B3_SynthBass.wav'
-rw-r--r-- 1 root root 8.1M Nov 13 11:30 'FirstBatch - MajorC3-B4_Accordian.wav'
-rw-r--r-- 1 root root 8.1M Nov 13 11:30 'FirstBatch - MajorC3-B4_AcousticGuitar.wav'
-rw-r--r-- 1 root root 8.1M Nov 13 11:30 'FirstBatch - MajorC3-B4_MetalGuitar_1stInv.wav'
-rw-r--r-- 1 root root 8.1M Nov 13 11:30 'FirstBatch - MajorC3-B4_MetalGuitar_2ndInv.wav'
-rw-r--r-- 1 root root 8.1M Nov 13 11:30 'FirstBatch - MajorC3-B4_MetalGuitar.wav'
-rw-r--r-- 1 root root 8.1M Nov 13 11:30 'FirstBatch - MajorC3-B4_Organ_1stInv.wav'
-rw-r--r-- 1 root root 8.1M Nov 13 11:30 'FirstBatch - MajorC3-B4_Organ_2ndInv.wav'
-rw-r--r-- 1 root root 8.1M Nov 13 11:30 'FirstBatch - MajorC3-B4_Piano_1stInv.wav'
-rw-r--r-- 1 root root 8.1M Nov 13 11:30 'FirstBatch - MajorC3-B4_Piano_2ndInv.wav'
-rw-r--r-- 1 root root 8.1M Nov 

In [ ]:
import os, glob
import numpy as np
import librosa

In [5]:
# ---------------- Config ----------------
dataset_dir = '/content/Chords_Dataset/WavFiles'
sr = 48000
bpm = 120
beat = int(sr * 60 / bpm)             # 24000 samples
interval_samples = beat * 4           # 3 beats played + 1 rest
active_samples = beat * 3
num_chords = 24
quality_mapping = {'Major': 0, 'Minor': 1}

# FFT window: bin spacing = sr / N = 1.953125 Hz  ->  bins 1..2048 = 1.95..4000 Hz exactly
N = 24576
n_bins = 2048
assert abs(sr / N * n_bins - 4000) < 1e-9
hop = beat // 2
window = np.hanning(N).astype(np.float32)
bin_freqs = np.arange(1, n_bins + 1) * sr / N

GROUP_BY = 'file'     # 'file' or 'instrument' (stricter)




In [6]:
# ---------------- NEW: bin -> pitch-class map ----------------
# Every FFT bin from PC_FMIN up is assigned to its nearest semitone (C=0 ... B=11, A4=440 Hz).
# Below ~100 Hz a semitone is only ~3 bins wide, so those bins are ignored (the harmonics
# of low notes still land above 100 Hz).
PC_FMIN = 100.0
semitones_from_c4 = 12 * np.log2(bin_freqs / 261.6256)
pc_of_bin = np.round(semitones_from_c4).astype(int) % 12
valid = bin_freqs >= PC_FMIN
P = np.zeros((n_bins, 12), dtype=np.float32)
P[np.arange(n_bins)[valid], pc_of_bin[valid]] = 1.0


def fft_features(seg):
    """2048 magnitude bins from 1.953 Hz to 4000 Hz. No interpolation."""
    spec = np.abs(np.fft.rfft(seg * window))[1:n_bins + 1]
    spec = spec / max(spec.max(), 1e-8)
    spec = np.log1p(100.0 * spec) / np.log1p(100.0)
    return spec.astype(np.float32)



In [7]:

# NEW: energy per pitch class, summed from the RAW (linear) FFT magnitudes
def pc_features(seg):
    """12 values: spectral energy at C, C#, ... B (all octaves folded together)."""
    mag = np.abs(np.fft.rfft(seg * window))[1:n_bins + 1]
    pc = mag @ P
    pc = pc / max(pc.max(), 1e-8)
    pc = np.log1p(100.0 * pc) / np.log1p(100.0)
    return pc.astype(np.float32)


# NEW: rotate the pitch-class vector so the root sits at index 0
def root_relative(pc, root):
    """
    pc: (B, 12), root: (B,) integers 0..11.
    Output index k = energy at (root + k) semitones:
      k=0 root, k=3 minor third, k=4 major third, k=7 fifth.
    After this, C minor and F# minor look the same to the quality network.
    """
    idx = (np.asarray(root)[:, None] + np.arange(12)[None, :]) % 12
    return np.take_along_axis(pc, idx, axis=1)

In [ ]:
# ---------------- Build dataset: First Batch ----------------
X_fft, X_pc, y_root, y_quality, groups = [], [], [], [], []
group_ids = {}

wav_files = sorted(glob.glob(os.path.join(dataset_dir, 'FirstBatch*.wav')))
for fi, path in enumerate(wav_files):
    name = os.path.basename(path).lower()
    quality = next((v for k, v in quality_mapping.items() if k.lower() in name), None)
    if quality is None:
        print('Skipping', name)
        continue

    if GROUP_BY == 'instrument':
        group_key = name.replace('.wav', '').split('_')[1]
    else:
        group_key = name
    gid = group_ids.setdefault(group_key, len(group_ids))

    y, _ = librosa.load(path, sr=sr, mono=True)

    for i in range(num_chords):
        start = i * interval_samples
        if start + active_samples > len(y):
            break
        active = y[start:start + active_samples]
        root = i % 12

        for s in range(0, active_samples - N + 1, hop):
            seg = active[s:s + N]
            X_fft.append(fft_features(seg))
            X_pc.append(pc_features(seg))                  # NEW
            y_root.append(root)
            y_quality.append(quality)
            groups.append(gid)

X_fft = np.array(X_fft); X_pc = np.array(X_pc)
y_root = np.array(y_root); y_quality = np.array(y_quality); groups = np.array(groups)
print('X_fft:', X_fft.shape, ' X_pc:', X_pc.shape)



X_fft: (2304, 2048)  X_pc: (2304, 12)


In [9]:
# ---------------- NEW: SecondBatch (real guitar recordings) ----------------
import re

PROGRESSIONS = {
    'maj': [(0, 0), (5, 0), (7, 0), (9, 1)],   # I  - IV - V  - vi
    'min': [(0, 1), (5, 1), (7, 1), (8, 0)],   # i  - iv - v  - VI
}
NOTE_PC = {'C': 0, 'D': 2, 'E': 4, 'F': 5, 'G': 7, 'A': 9, 'B': 11}

CHORD_SEC  = 2.0
LEAD_IN    = 0.0
TRIM_START = 0.15
TRIM_END   = 0.20

def second_batch_chords(filename):
    m = re.match(r'secondbatch - ([a-g])([#b]?)(maj|min)guitar\.wav$', filename.lower())
    letter, acc, mode = m.groups()
    key = (NOTE_PC[letter.upper()] + {'#': 1, 'b': -1, '': 0}[acc]) % 12
    loop = [((key + off) % 12, q) for off, q in PROGRESSIONS[mode]]
    return loop * 2

def build_second_batch(wav_dir):
    Xf, Xp, yr, yq, gr = [], [], [], [], []
    for path in sorted(glob.glob(os.path.join(wav_dir, 'SecondBatch*.wav'))):
        name = os.path.basename(path).lower()
        gid = group_ids.setdefault(name, len(group_ids))
        y, _ = librosa.load(path, sr=sr, mono=True)

        for i, (root, quality) in enumerate(second_batch_chords(name)):
            a = int((LEAD_IN + i * CHORD_SEC + TRIM_START) * sr)
            b = int((LEAD_IN + (i + 1) * CHORD_SEC - TRIM_END) * sr)
            active = y[a:b]
            for s in range(0, len(active) - N + 1, hop):
                seg = active[s:s + N]
                Xf.append(fft_features(seg)); Xp.append(pc_features(seg))
                yr.append(root); yq.append(quality); gr.append(gid)
    return (np.array(Xf), np.array(Xp), np.array(yr), np.array(yq), np.array(gr))

X_fft2, X_pc2, y_root2, y_quality2, groups2 = build_second_batch(dataset_dir)
print('SecondBatch:', X_fft2.shape, X_pc2.shape)

# Combine with the first batch
source    = np.concatenate([np.zeros(len(X_fft)), np.ones(len(X_fft2))])   # 0 = MIDI, 1 = guitar
X_fft     = np.concatenate([X_fft, X_fft2])
X_pc      = np.concatenate([X_pc, X_pc2])
y_root    = np.concatenate([y_root, y_root2])
y_quality = np.concatenate([y_quality, y_quality2])
groups    = np.concatenate([groups, groups2])
print('Combined:', X_fft.shape)

SecondBatch: (480, 2048) (480, 12)
Combined: (2784, 2048)
